In [0]:
%pip install lightgbm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from joblib import Parallel, delayed

LOCATIONS       = list(range(48))
FORECAST_DATE   = pd.to_datetime('2010-08-14')
HORIZON_STEPS   = 288
ALPHA           = 0.5
NUM_WORKERS     = 16
LGBM_THREADS    = 1

date_str = FORECAST_DATE.strftime('%Y-%m-%d')
best_params_df = pd.read_csv(
    f'/dbfs/mnt/thesis/models/lgbm/basic/best_params.csv'
    ,parse_dates=['TUNING_DATE']
)
best_params_df = best_params_df.loc[best_params_df.groupby('LOCATION')['TUNING_DATE'].idxmax()]
best_params_df.drop(['TUNING_DATE'], axis=1, inplace=True)

input_path = '/dbfs/mnt/thesis/output_data/'
df_raw = (
    pd.read_csv(input_path + "processed_data.csv", parse_dates=['DATETIME'])
      .sort_values(['LOCATION', 'DATETIME'])
      .reset_index(drop=True)
)

def create_features_trimmed(df):
    df = df.copy()
    # cyclical hour features
    df['hour']      = df['DATETIME'].dt.hour
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    # categorical time features
    df['day_of_week'] = df['DATETIME'].dt.dayofweek.astype('category')
    df['month']       = df['DATETIME'].dt.month.astype('category')
    # key lag features and roll
    for lag in (1, 12, 288, 2016):
        df[f'lag_{lag}'] = df.groupby('LOCATION')['VALUE'].shift(lag)
    grp = df.groupby('LOCATION')['VALUE']
    df['roll_mean_288'] = grp.transform(lambda x: x.shift(1).rolling(288).mean())
    df['roll_std_288']  = grp.transform(lambda x: x.shift(1).rolling(288).std(ddof=1))
    df.dropna(inplace=True)
    return df

#generate features and cast LOCATION to category
df_features = create_features_trimmed(df_raw)
df_features['LOCATION'] = df_features['LOCATION'].astype('category')

FEATURE_COLS = [
    'hour_sin','hour_cos','day_of_week','month',
    'lag_1','lag_12','lag_288','lag_2016',
    'roll_mean_288','roll_std_288',
    'LOCATION'
]
CATEGORICAL_FEATURES = ['LOCATION','day_of_week','month']
TARGET_COL = 'VALUE'

In [0]:
#Per‑location forecasting function
def process_location(loc):
    # load best params for this location
    print("Processing location:", loc)
    row = best_params_df.query("LOCATION == @loc").iloc[0].to_dict()
    for k in ('LOCATION','BEST_MAE'): row.pop(k, None)
    for ip in ('num_leaves','max_depth','min_child_samples','n_estimators'):
        if ip in row:
            row[ip] = int(row[ip])

    common_params = {
        **row,
        # 'objective':   'quantile',
        # 'alpha':       ALPHA,
        # 'metric':      'quantile',
        'objective':   'regression',
        'metric':      'mae',
        'verbosity':   -1,
        'n_jobs':      LGBM_THREADS
    }

    hist_feat = df_features[
        (df_features.LOCATION == loc) &
        (df_features.DATETIME  <  FORECAST_DATE)
    ]
    X_hist = hist_feat[FEATURE_COLS]
    y_hist = hist_feat[TARGET_COL]

    # train two ensemble models
    m1 = lgb.LGBMRegressor(**{**common_params, 'random_state':  42})
    m2 = lgb.LGBMRegressor(**{**common_params, 'random_state': 2025})

    m1.fit(X_hist, y_hist,
           categorical_feature=CATEGORICAL_FEATURES,
           callbacks=[lgb.log_evaluation(period=0)])
    m2.fit(X_hist, y_hist,
           categorical_feature=CATEGORICAL_FEATURES,
           callbacks=[lgb.log_evaluation(period=0)])

    importances = [
        {'LOCATION': loc, 'FEATURE': f, 'IMPORTANCE': imp}
        for f, imp in zip(FEATURE_COLS, m1.booster_.feature_importance('gain'))
    ]

    raw_loc = df_raw[df_raw.LOCATION == loc].sort_values('DATETIME')
    last = raw_loc[raw_loc.DATETIME < FORECAST_DATE]
    last_vals  = last['VALUE'].tolist()
    last_times = last['DATETIME'].tolist()

    preds = []
    for _ in range(HORIZON_STEPS):
        nt = last_times[-1] + pd.Timedelta(minutes=5)
        d = {
            'hour_sin':    np.sin(2 * np.pi * nt.hour / 24),
            'hour_cos':    np.cos(2 * np.pi * nt.hour / 24),
            'day_of_week': nt.dayofweek,
            'month':       nt.month,
            'LOCATION':    loc
        }
        # lags and roll
        for lag in (1,12,288,2016):
            d[f'lag_{lag}'] = last_vals[-lag]
        window = last_vals[-288:]
        d['roll_mean_288'] = np.mean(window)
        d['roll_std_288']  = np.std(window, ddof=1)

        # build DataFrame and align categories
        Xf = pd.DataFrame([d])[FEATURE_COLS]
        Xf['LOCATION'] = Xf['LOCATION'] \
            .astype('category') \
            .cat.set_categories(df_features['LOCATION'].cat.categories)
        Xf['day_of_week'] = Xf['day_of_week'] \
            .astype('category') \
            .cat.set_categories(df_features['day_of_week'].cat.categories)
        Xf['month'] = Xf['month'] \
            .astype('category') \
            .cat.set_categories(df_features['month'].cat.categories)

        p1 = m1.predict(Xf)[0]
        p2 = m2.predict(Xf)[0]
        p  = 0.5 * (p1 + p2)

        preds.append({'LOCATION': loc, 'DATETIME': nt, 'PREDICTED': p})
        last_times.append(nt)
        last_vals.append(p)

    return pd.DataFrame(preds), importances

In [0]:
#Parallel execution
print(f"Launching parallel forecasting for {len(LOCATIONS)} locations...")
results = Parallel(n_jobs=NUM_WORKERS, backend='threading')(
    delayed(process_location)(loc) for loc in LOCATIONS
)

# combine results
all_forecasts = pd.concat([r[0] for r in results], ignore_index=True)
importances_df = pd.DataFrame([imp for r in results for imp in r[1]])

print("Parallel daily forecasting complete.")

out_dir = '/dbfs/mnt/thesis/predictions/lgbm/ensemble_rolling_periodic_retune/'
all_forecasts.to_csv(f'{out_dir}predictions_{date_str}.csv', index=False)
# importances_df.to_csv(f'{out_dir}daily_feature_importances_{date_str}.csv', index=False)

Launching parallel forecasting for 48 locations...
Processing location: 0
Processing location: 1
Processing location: 2
Processing location: 3
Processing location: 4
Processing location: 5
Processing location: 6
Processing location: 7
Processing location: 8
Processing location: 9
Processing location: 10
Processing location: 11
Processing location: 12
Processing location: 13
Processing location: 14
Processing location: 15
Processing location: 16
Processing location: 17
Processing location: 18
Processing location: 19
Processing location: 20
Processing location: 21
Processing location: 22
Processing location: 23
Processing location: 24
Processing location: 25
Processing location: 26
Processing location: 27
Processing location: 28
Processing location: 29
Processing location: 30
Processing location: 31
Processing location: 32
Processing location: 33
Processing location: 34
Processing location: 35
Processing location: 36
Processing location: 37
Processing location: 38
Processing location: 39